<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c37-header.png" alt="Nextia Learning · Recommendation and Personalisation: Show Each Person What Matters" width="100%">

# Solution: Business rules and re-ranking

**[Business rules and re-ranking](https://learning.nextia-ai.com/courses/recommendations/m07/business-rules-and-re-ranking/)** · Recommendation and Personalisation: Show Each Person What Matters · Module 7, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find the products a shop must not suggest, apply business rules to a list, and measure what each rule costs.

| | |
|---|---|
| **Time** | About 20 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB). No account and no key. |
| **You should know** | Recall@10, from [Ranking metrics](https://learning.nextia-ai.com/courses/recommendations/m03/ranking-metrics/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, SciPy 1.18.1 and matplotlib 3.11.2 (implicit 0.7.3 or LightGBM 4.7.0 where the lesson needs it), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m07/business-rules-and-re-ranking/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/M07-L03-business-rules-and-re-ranking/business-rules-and-re-ranking-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, scipy 1.18.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 scipy==1.18.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'scipy']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 7 MB) into a folder `larkfield-recs` and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-recs` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop, and its shop log is **synthetic**: our code made the products, the customers and every view, basket and order, with a model of customer behaviour. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/data/dataset-card.md) explains how.

In [ ]:
# Setup: download the files into larkfield-recs/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C37/"
FILES = {
    "data/products.csv": "3da37a3809f6067da12eb5778b8abcf809545599db6959208c72a790861407ff",
    "data/events.csv.gz": "3f46ddfff683586ab1548bc0e6756d85a439a1de941e6de3030edc383d06d1b3",
    "data/splits.json": "c58597803ced6fa787ff23c86d8ee6e9f0c1d7024c0ec109d2cca25caf6213e6",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-recs").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-recs and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-recs.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, SciPy's sparse matrices as `sp` and matplotlib. It defines the course's small helpers, the same ideas as the course project:

- `products()`, `customers()` and `events()` read the data. `events()` has one row per view, basket add or order, with the column `day` (days since 1 February 2025).
- `split(period)` gives the events before the period (`train`, to learn from) and, for each customer who ordered in the period, the products they ordered (`truth`). The periods are `"validation"`, `"test"` and `"launch"`.
- `recall_at_k(list, ordered, k)`, `ndcg_at_k(list, ordered, k)` and `coverage(lists, n_products)` measure suggestion lists.
- `show(ids)` prints products with their title and subcategory.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the time split, the measures, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
START = pd.Timestamp("2025-02-01")

def products():
    """One row per product (2,994)."""
    df = pd.read_csv("data/products.csv", keep_default_na=False)
    for col in ("age_restricted", "sensitive", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def customers():
    """One row per customer (40,000)."""
    return pd.read_csv("data/customers.csv")

def events():
    """One row per event: view, basket or order. day = days since 1 February 2025."""
    ev = pd.read_csv("data/events.csv.gz", parse_dates=["timestamp"])
    ev["day"] = (ev["timestamp"] - START).dt.days
    return ev

def split(period, ev=None):
    """train = the events before the period; truth = {customer: set of products ordered in the period}."""
    ev = events() if ev is None else ev
    first, last = json.load(open("data/splits.json"))[period]
    a, b = (pd.Timestamp(first) - START).days, (pd.Timestamp(last) - START).days
    train = ev[ev["day"] < a]
    target = ev[(ev["day"] >= a) & (ev["day"] <= b) & (ev["event"] == "order")]
    truth = target.groupby("customer_id")["product_id"].agg(set).to_dict()
    return train, truth

def recall_at_k(rec, ordered, k=10):
    """Share of the ordered products that are in the first k suggestions."""
    return len(set(list(rec)[:k]) & set(ordered)) / len(ordered)

def ndcg_at_k(rec, ordered, k=10):
    """DCG of the list divided by the best possible DCG (discount log2(place + 1))."""
    gains = [1.0 if p in ordered else 0.0 for p in list(rec)[:k]]
    dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
    ideal = sum(1 / np.log2(i + 2) for i in range(min(len(ordered), k)))
    return dcg / ideal

def coverage(lists, n_products, k=10):
    """Share of the catalogue that appears in at least one list."""
    return len({p for rec in lists for p in list(rec)[:k]}) / n_products

def rounded(x, n=3):
    """x rounded to n places (None or ... unchanged: the task is not done yet)."""
    return x if x is None or x is ... else round(float(x), n)

def show(ids, n=10):
    """Print products: place, ID, title, subcategory."""
    global _PR
    if "_PR" not in globals():
        _PR = products().set_index("product_id")
    for i, p in enumerate(list(ids)[:n], 1):
        print(f"{i:2}. {p}  {_PR.loc[p, 'title'][:52]:<52}  {_PR.loc[p, 'subcategory']}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. What a popular list should not show

A list ranked by interest knows nothing about stock, age limits or privacy. Here is the shop's all-time popular list, with the reasons why the shop should not show some of its products.

> **Predict.** Will you find a product that is out of stock, or a health product, in the top 10?

In [ ]:
ev = events()
pr = products().set_index("product_id")
train, truth = split("test", ev)
first_day = train["day"].max() + 1
orders = train[train["event"] == "order"]
popular = orders.groupby("product_id")["customer_id"].nunique().sort_values(ascending=False, kind="stable")

def problems(p):
    """The reasons why a shop should not show product p (an empty list = no problem)."""
    out = []
    if not pr.loc[p, "in_stock"]:
        out.append("out of stock")
    if pr.loc[p, "age_restricted"]:
        out.append("alcohol")
    if pr.loc[p, "sensitive"]:
        out.append("sensitive")
    return out

print("The all-time popular list before the test period:")
for i, p in enumerate(popular.index[:10], 1):
    print(f"{i:2}. {pr.loc[p, 'title'][:44]:<44} {pr.loc[p, 'subcategory']:<22} {', '.join(problems(p))}")

## 2. A personal list, before the rules

The project applies the rules to its ranker's lists. To keep this notebook small, we use a simple personal list instead: each customer's products of the last 30 days, newest first, then trending. Like the ranker, it puts recent products first. Each customer gets 50 products, so that removed products can be replaced. It takes a few seconds.

In [ ]:
# A simple personal list of 50: the customer's products of the last 30 days (newest first), then trending.
buyers28 = orders[orders["day"] >= first_day - 28].groupby("product_id")["customer_id"].nunique()
trending = list(buyers28.sort_values(ascending=False, kind="stable").index)
recent = train[train["day"] >= first_day - 30].sort_values("day", ascending=False)
recent_by_customer = recent.groupby("customer_id")["product_id"].agg(lambda s: list(dict.fromkeys(s)))
long_lists = {}
for c in truth:
    lst = list(dict.fromkeys(recent_by_customer.get(c, [])[:30] + trending))
    long_lists[c] = lst[:50]

# Products people buy again (compost, pet food ...) have repeat_days > 0: buying them before is fine.
repeat = pr["repeat_days"] > 0
once = orders[~orders["product_id"].map(repeat).to_numpy()]
bought = once.groupby("customer_id")["product_id"].agg(set).to_dict()
blocked = set(pr.index[~pr["in_stock"] | pr["age_restricted"] | pr["sensitive"]])

top10 = {c: lst[:10] for c, lst in long_lists.items()}
with_bought = np.mean([bool(set(l) & bought.get(c, set())) for c, l in top10.items()])
print(f"{len(top10):,} test customers; lists with an already-bought product: {with_bought:.1%}")
print(f"places with a blocked product (out of stock, alcohol or sensitive): "
      f"{np.mean([p in blocked for l in top10.values() for p in l]):.1%}")

## 3. Your turn: the already-bought rule

Write the rule that removes the products a customer already bought. `bought` holds, for each customer, the products they ordered before the test period, without the products people buy again.

In [ ]:
def remove_already_bought(lst, customer):
    have = bought.get(customer, set())
    return [p for p in lst if p not in have]

c0 = "C106777"
print(len(long_lists[c0]), "->", len(remove_already_bought(long_lists[c0], c0)))

## 4. Each rule alone, then all rules

The function `apply` uses the rules in the project's order: remove blocked products, remove already-bought products, keep at most 3 per subcategory, cut to 10, and fill empty places from trending with the same rules. It takes about half a minute.

In [ ]:
subcategory = pr["subcategory"].to_dict()

def apply(lst, customer, block=True, already=True, cap=3, k=10):
    """Remove blocked and already-bought products, keep at most `cap` per subcategory, cut to k, fill from trending."""
    out, per_sub = [], {}
    candidates = remove_already_bought(lst + trending, customer) if already else lst + trending
    for p in candidates:
        if p in out or (block and p in blocked):
            continue
        sub = subcategory[p]
        if per_sub.get(sub, 0) >= cap:
            continue
        per_sub[sub] = per_sub.get(sub, 0) + 1
        out.append(p)
        if len(out) == k:
            break
    return out

def score(make):
    lists = {c: make(c) for c in truth}
    return (np.mean([recall_at_k(lists[c], truth[c]) for c in truth]),
            np.mean([bool(set(lists[c]) & bought.get(c, set())) for c in truth]))

if remove_already_bought(["P0"], "nobody") is not ...:
    table = {
        "no rules": score(lambda c: long_lists[c][:10]),
        "only blocked products": score(lambda c: apply(long_lists[c], c, already=False, cap=99)),
        "only already bought": score(lambda c: apply(long_lists[c], c, block=False, cap=99)),
        "only 3 per subcategory": score(lambda c: apply(long_lists[c], c, block=False, already=False)),
        "all rules": score(lambda c: apply(long_lists[c], c)),
    }
    for name, (r, b) in table.items():
        print(f"{name:<24} recall@10 {r:.3f}   lists with an already-bought product {b:6.1%}")
    recall_all = table["all rules"][0]
else:
    recall_all = None
    print("Write remove_already_bought in the cell above first.")

In [ ]:
expect('recall_all', rounded(recall_all, 3), 0.173, tolerance=0.002)

> **Check.** Without rules, 30.5% of these lists have an already-bought product, and recall@10 is 0.182. The already-bought rule alone gives 0.174, and all rules 0.173. In the project, all rules cost the two-stage ranker 0.019 (0.234 to 0.215): `recs rules two-stage`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Sessions, sequences and LLMs, briefly](https://learning.nextia-ai.com/courses/recommendations/m07/sessions-sequences-and-llms-briefly/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/recommendations/m07/business-rules-and-re-ranking/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield shop log is synthetic: our code made the products, the customers and everything they did, with a model of customer behaviour. No real customer is in this data.